In [1]:
import pandas as pd
from functools import partial

In [2]:
def build_panel_dataset(df, freq, min_obs, indiv_col='user',date_col='date',metrics=None):

    data = df.copy()

    data[date_col] = pd.to_datetime(data[date_col])
    data["period"] = data[date_col].dt.to_period(freq)

    counts = (
        data
        .groupby([indiv_col, "period"])
        .size()
        .reset_index(name="n_obs")
    )
    valid_groups = counts[counts["n_obs"] >= min_obs]
    data = data.merge(
        valid_groups[[indiv_col, "period", "n_obs"]],
        on=[indiv_col, "period"],
        how="inner"
    )

    rows = []
    grouped = data.groupby([indiv_col, "period"])
    for (user, period), group in grouped:
        row = {
            indiv_col: user,
            "period": period,
            "n_obs": len(group)
        }
        # Calcul des métriques custom
        for metric_name, metric_func in metrics.items():
            try:
                row[metric_name] = metric_func(group)
            except Exception as e:
                row[metric_name] = pd.NA
                print(
                    f"Erreur pour {metric_name} "
                    f"({user}, {period}) : {e}"
                )
        rows.append(row)

    panel_df = pd.DataFrame(rows)
    return panel_df

In [7]:
# Définition des mesures
def left_right_side(df):
    if df['user_political'].sum()<3:
        return "Apolitical"
    else :
        if len(df["user_left_right"].mode())>1:
            return "Apolitical"
        else:
            return df["user_left_right"].mode().iloc[0]

def polarisation_self_side(df, user_side):
    y = df[(df['inter_left_right']==user_side) | ((df['user_left_right']==user_side) & (df['inter_left_right'].isna()))]
    if len(y)==0: # If no interactions with same side
        prop_violent_vs_same = 0
    else :
        prop_violent_vs_same = y['user_violent'].sum()/len(y)
    return prop_violent_vs_same*100

def polarisation_other_side(df, other_side):
    x = df[df['inter_left_right']==other_side]
    if len(x)==0: # If no interactions with other side
        prop_violent_vs_other = 0
    else :
        prop_violent_vs_other = x['user_violent'].sum()/len(x) 
    return prop_violent_vs_other*100

def homophilie(df, user_side):
    other_side = [l for l in ['Left', 'Right'] if l!=user_side][0]
    x = df[(df['inter_left_right']==other_side) | (df['inter_left_right']==user_side) | ((df['user_left_right']==user_side) & (df['inter_left_right'].isna()))]
    z = df[(df['inter_political']==1) | (df['user_political']==1)]
    if len(z)>0:
        return len(x)*100/len(z)
    else: 
        return 0

In [8]:
metrics = {

    # Nombre d'observations politique
    "user_total_political_interactions": lambda g: g["user_political"].sum(),

    # bord politique
    "user_period_side": left_right_side,

    # Polarisation brute
    'user_polarisation_left' : partial(polarisation_other_side, other_side='Right'),
    'user_polarisation_right' : partial(polarisation_other_side, other_side='Left'),

    # Polarisation dans son propre camp
    'user_polarization_self_left' : partial(polarisation_self_side, user_side='Left'),
    'user_polarization_self_right' : partial(polarisation_self_side, user_side='Right'),

    # Homophilie
    'user_homophilie_left' : partial(homophilie, user_side='Left'),
    'user_homophilie_right' : partial(homophilie, user_side='Right'),
    }

In [9]:
df = pd.read_csv('../clean_data/struct_annotated_interactions.csv')

In [10]:
# Construction du panel
panel_df = build_panel_dataset(
    df,
    freq="M",
    min_obs=10,
    metrics=metrics
)

C:\Users\cfrou\AppData\Local\Temp\ipykernel_36276\4230742483.py:6: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  data["period"] = data[date_col].dt.to_period(freq)


In [11]:
panel_df['user_polarization']=0
panel_df.loc[panel_df['user_period_side']=='Left', 'user_polarization'] = panel_df['user_polarisation_left']
panel_df.loc[panel_df['user_period_side']=='Right', 'user_polarization'] = panel_df['user_polarisation_right']

panel_df['user_net_polarization']=0
panel_df.loc[panel_df['user_period_side']=='Left', 'user_net_polarization'] = panel_df['user_polarisation_left'] - panel_df['user_polarization_self_left']
panel_df.loc[panel_df['user_period_side']=='Right', 'user_net_polarization'] = panel_df['user_polarisation_right'] - panel_df['user_polarization_self_right']


panel_df['user_homophilie']=0
panel_df.loc[panel_df['user_period_side']=='Left', 'user_homophilie'] = panel_df['user_homophilie_left']
panel_df.loc[panel_df['user_period_side']=='Right', 'user_homophilie'] = panel_df['user_homophilie_right']

# Création des indicatrices de bord politiques
panel_df['user_Left'] = (panel_df['user_period_side']=='Left').astype(int)
panel_df['user_Right'] = (panel_df['user_period_side']=='Right').astype(int)

# Suppression des colonnes inutiles
panel_df = panel_df.drop(columns=['user_polarisation_left', 'user_polarisation_right',
       'user_polarization_self_left', 'user_polarization_self_right',
       'user_homophilie_left', 'user_homophilie_right', 'user_period_side'])

C:\Users\cfrou\AppData\Local\Temp\ipykernel_36276\3395480192.py:2: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[0. 0. 0. ... 0. 0. 0.]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  panel_df.loc[panel_df['user_period_side']=='Left', 'user_polarization'] = panel_df['user_polarisation_left']
C:\Users\cfrou\AppData\Local\Temp\ipykernel_36276\3395480192.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[ -7.14285714 -22.22222222   0.         ... -28.57142857 -19.04761905
  -5.        ]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  panel_df.loc[panel_df['user_period_side']=='Left', 'user_net_polarization'] = panel_df['user_polarisation_left'] - panel_df['user_polarization_self_left']
C:\Users\cfrou\AppData\Local\Temp\ipykernel_36276\33954

### Panel sans saut par série

On retrouvera simplement le panel sans remplissage en excluant les lignes avec un nombre de messages observés nul.

In [14]:
# Remplissage de tous les sauts par individu

panel_df = panel_df.sort_values(['user', 'period'])
print(len(panel_df))

all_periods = (
    panel_df.groupby('user')
      .apply(lambda g: pd.period_range(start=g['period'].min(),
                                       end=g['period'].max(),
                                       freq='M'))
      .explode()
      .reset_index()
      .rename(columns={0: 'period'})
)

df_full = all_periods.merge(panel_df, on=['user', 'period'], how='left')

cols_to_fill = panel_df.columns.difference(['user', 'period'])
df_full[cols_to_fill] = df_full[cols_to_fill].fillna(0)
print(len(df_full))

11262
24476


C:\Users\cfrou\AppData\Local\Temp\ipykernel_36276\3050041692.py:8: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.period_range(start=g['period'].min(),


In [15]:
# Export en csv
df_full.to_csv('../clean_data/full_panel_interactions.csv', index=False)